In [1]:
FAMILY_MAP = {
    "Gaussian": "GaussianCopula",
    "Bernoulli": "BernoulliCopula",
    "NegBin": "NegBinCopula",
    "NegBinIRLS": "NegBinIRLSCopula",
    "Poisson": "PoissonCopula",
    "pNMF": "PositiveNMF",
    "ZeroInflatedNegBin": "ZeroInflatedNegBinCopula",
    "ZeroInflatedPoisson": "ZeroInflatedPoissonCopula"
}

In [2]:
import importlib
import random

def make_formulas(formulas: dict) -> dict:
    formula_strs = {}

    # loop over model parameters
    for param, variables in formulas.items():
        terms = []

        # loop over predictors for that parameter
        for var in variables:
            if "transform" in var and var["transform"].get("type") == "spline":
                terms.append(f"bs({var['variable']}, df={var['transform']['df']})")
            else:
                terms.append(var['variable'])
        formula_strs[f"{param}_formula"] = "~ " + " + ".join(terms) if terms else "~ 1"

    return formula_strs

def generate_data(state):
    # prepare the formula and simulator type
    class_name = FAMILY_MAP[state["family"]]
    simulator_class = getattr(importlib.import_module("scdesigner.simulators"), class_name)
    fmla = make_formulas(state["formulas"])

    # fit the simulator
    simulator = simulator_class(**fmla)
    simulator.fit(state["template"], max_epochs=25)

    # sample new data
    n_cells = state["n_cells"] or len(state["template"])
    ix = random.choices(range(len(state["template"])), k=n_cells)
    new_data = simulator.sample(state["template"].obs.iloc[ix, :])

    return simulator.parameters, new_data

Here is a version of a negative binomial model where we use cell_type as a predictor and don't have any change as a function of pseudotime.

In [3]:
from scdesigner.datasets import pancreas

formulas = {
    "mean": [{"variable": "cell_type"}],
    "dispersion": [{"variable": "cell_type"}],
    "copula": []
}

example_sce = pancreas()
state = {"template": example_sce, "family": "NegBin", "formulas": formulas, "n_cells": None}
parameters, adata_sim = generate_data(state)

Epoch 25/25, Loss: 1.9644, Val Loss: 1.9669


Estimating copula correlation: 100%|██████████| 3/3 [00:00<00:00, 18.12it/s]


Here are the coefficients for each gene's negative binomial model.

In [4]:
parameters["marginal"]["mean"]

,Pyy,Iapp,Chgb,Rbp4,Spp1,Chga,Cck,Ins1,Nnat,Ins2,...,Nkx6-1,Fxyd3,Hn1,Smarcd2,Pdia6,Ffar2,Hes6,Serpinh1,Npy,1110012L19Rik
Intercept,0.056393,-0.913212,-1.434003,-0.371312,4.194845,-2.053644,-1.904621,-1.275848,-1.547781,-2.636792,...,0.478958,-0.248252,1.503538,0.253186,1.158546,-0.605932,0.406455,1.610021,-4.507418,-0.713368
cell_type[T.Ngn3 high EP],-0.339077,-0.154239,1.118734,-0.881085,-1.575483,2.066646,4.854985,-0.410084,1.006965,-0.193357,...,0.687333,0.891716,0.340465,1.210708,-0.411229,1.172308,1.272929,-0.248277,-0.091334,1.328641
cell_type[T.Pre-endocrine],3.050847,0.409684,5.192988,2.746677,-5.625076,5.042534,4.604670,-0.320688,1.032740,-0.026869,...,0.767221,1.301734,-0.597094,0.081594,-1.151457,1.760671,0.142529,-1.836462,-0.324963,1.572527
cell_type[T.Beta],4.887130,5.907669,4.737278,4.140733,-5.543490,4.952433,2.914708,5.442261,5.048255,6.070582,...,1.024900,0.887283,-1.000395,-1.103323,0.740540,0.999349,-0.187658,-2.099929,4.552769,-0.011510


Here's the estimated copula covariance matrix.

In [5]:
parameters["copula"]["Intercept"]

,Pyy,Iapp,Chgb,Rbp4,Spp1,Chga,Cck,Ins1,Nnat,Ins2,...,Nkx6-1,Fxyd3,Hn1,Smarcd2,Pdia6,Ffar2,Hes6,Serpinh1,Npy,1110012L19Rik
Pyy,1.000000,0.126321,-0.085389,0.346678,0.000623,-0.035735,-0.010717,0.005677,-0.000792,0.021791,...,-0.102327,0.025770,-0.023785,-0.064202,0.080200,-0.034890,-0.039160,0.011005,0.040048,-0.037759
Iapp,0.126321,1.000000,-0.067082,0.088181,0.032738,-0.042089,-0.006166,0.217170,0.208065,0.263073,...,0.076002,-0.030533,-0.000354,-0.047102,0.216239,-0.075577,-0.008646,0.010124,0.125227,-0.043758
Chgb,-0.085389,-0.067082,1.000000,-0.026880,-0.066219,0.268039,0.035865,-0.088098,-0.065218,-0.064559,...,0.139615,0.110847,-0.049165,-0.081802,-0.061537,0.129175,-0.001272,-0.106894,-0.023548,0.066967
Rbp4,0.346678,0.088181,-0.026880,1.000000,0.015750,-0.011822,-0.079656,0.026787,0.001032,0.028769,...,-0.109846,0.069343,-0.079439,-0.110310,0.102679,-0.028936,-0.082720,-0.019946,0.015296,-0.083640
Spp1,0.000623,0.032738,-0.066219,0.015750,1.000000,-0.101596,-0.314775,0.027572,-0.014754,0.039638,...,-0.145158,-0.122502,-0.153297,-0.119933,0.181906,-0.063257,-0.036806,0.132208,0.065363,-0.150834
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Ffar2,-0.034890,-0.075577,0.129175,-0.028936,-0.063257,0.075138,0.090956,-0.068157,-0.077033,-0.040072,...,0.063466,0.118909,0.049447,0.063053,-0.038642,1.000000,0.053995,-0.038933,-0.067653,0.079968
Hes6,-0.039160,-0.008646,-0.001272,-0.082720,-0.036806,-0.027054,0.099431,-0.015931,0.029630,0.043839,...,0.090323,0.046832,0.237574,0.208088,0.013306,0.053995,1.000000,0.125868,-0.023937,0.099290
Serpinh1,0.011005,0.010124,-0.106894,-0.019946,0.132208,-0.085992,-0.029926,0.019398,0.027503,0.018689,...,-0.014499,-0.035117,0.115858,0.135863,0.162780,-0.038933,0.125868,1.000000,0.007564,0.012198
Npy,0.040048,0.125227,-0.023548,0.015296,0.065363,-0.002341,0.010873,0.136334,0.156191,0.140886,...,0.047979,-0.043304,0.007188,-0.012434,0.133251,-0.067653,-0.023937,0.007564,1.000000,-0.010460


We can let the copula correlations depend on the cell type.

In [6]:
formulas["copula"] = [{"variable": "cell_type"}]
state["formulas"] = formulas
parameters, _ = generate_data(state)

# covariance matrix for this cell type
parameters["copula"]["cell_type[T.Beta]"]

Epoch 25/25, Loss: 1.9644, Val Loss: 1.9669


Estimating copula correlation: 100%|██████████| 3/3 [00:00<00:00, 24.25it/s]


,Pyy,Iapp,Chgb,Rbp4,Spp1,Chga,Cck,Ins1,Nnat,Ins2,...,Nkx6-1,Fxyd3,Hn1,Smarcd2,Pdia6,Ffar2,Hes6,Serpinh1,Npy,1110012L19Rik
Pyy,1.000000,0.207740,-0.098843,0.293167,0.001171,0.018371,0.207174,0.026863,0.055896,0.008526,...,-0.036037,0.025904,0.037427,-0.044273,0.074343,0.088801,0.014141,0.067883,-0.084031,0.011012
Iapp,0.207740,1.000000,-0.363959,0.160456,0.119042,-0.145963,0.035572,0.630846,0.733933,0.696559,...,0.279717,-0.114319,0.065675,-0.051819,0.594138,-0.189291,-0.007132,0.093320,0.363619,0.044982
Chgb,-0.098843,-0.363959,1.000000,-0.056308,-0.111537,0.297899,0.179310,-0.302399,-0.290686,-0.309793,...,0.010352,0.231488,0.086213,0.007014,-0.158622,0.207239,0.099362,-0.000488,-0.193257,0.001178
Rbp4,0.293167,0.160456,-0.056308,1.000000,-0.014920,0.106800,0.195990,-0.053400,0.120484,-0.010274,...,-0.031764,0.243617,0.120160,0.028258,0.150326,0.067519,0.027826,0.034498,-0.085550,0.085892
Spp1,0.001171,0.119042,-0.111537,-0.014920,1.000000,0.013465,-0.073730,0.125713,0.135131,0.155592,...,0.108096,-0.029671,0.022053,-0.005206,0.085419,-0.094031,0.036239,0.036901,0.114982,0.027449
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Ffar2,0.088801,-0.189291,0.207239,0.067519,-0.094031,0.129450,0.084995,-0.159847,-0.126356,-0.145581,...,0.008929,0.138065,0.055173,0.070706,-0.095036,1.000000,0.079395,0.021792,-0.112455,0.063407
Hes6,0.014141,-0.007132,0.099362,0.027826,0.036239,0.058406,0.102845,-0.071941,-0.073944,-0.073600,...,0.120978,0.067097,0.118050,0.116716,-0.038314,0.079395,1.000000,0.088120,-0.064679,0.170705
Serpinh1,0.067883,0.093320,-0.000488,0.034498,0.036901,0.061049,0.064859,0.097595,0.080651,0.170330,...,0.096081,0.057236,0.156663,0.087891,0.160695,0.021792,0.088120,1.000000,0.116181,0.082233
Npy,-0.084031,0.363619,-0.193257,-0.085550,0.114982,-0.077512,-0.011946,0.432437,0.436257,0.471613,...,0.168771,-0.098465,0.023193,0.043234,0.308148,-0.112455,-0.064679,0.116181,1.000000,-0.032519


We can allow the means and dispersions to vary smoothly over pseudotime by using spline transformations in the predictors. The `make_formulas` function transforms the object below into strings like `~ bs(pseudotime, df=8)`. I'm breaking that string down into this structure because I think it might map more easily onto user inputs (e.g., selecting variables from a list populated by the `obs` field of the anndata object, then clicking whether a spline should be used). But internally we just need the formula strings -- let me know if there's a more natural data structure from a UI perspective.

In [7]:
formulas = {
    "mean": [{"variable": "pseudotime", "transform": {"type": "spline", "df": 8}}],
    "dispersion": [{"variable": "pseudotime", "transform": {"type": "spline", "df": 3}}],
    "copula": [{"variable": "cell_type"}]
}

state = {"template": example_sce, "family": "NegBin", "formulas": formulas, "n_cells": None}
generate_data(state)

Epoch 25/25, Loss: 1.8605, Val Loss: 1.8677


Estimating copula correlation: 100%|██████████| 3/3 [00:00<00:00, 24.48it/s]


({'marginal': {'mean':                               Pyy      Iapp      Chgb      Rbp4      Spp1  \
   Intercept               -0.005292 -0.967646 -1.726450 -0.056083  4.388359   
   bs(pseudotime, df=8)[1] -0.225362 -0.343137  0.024613 -0.549686 -0.446189   
   bs(pseudotime, df=8)[2] -0.325008  0.731518 -0.899952 -0.933407  0.734141   
   bs(pseudotime, df=8)[3] -0.468245 -1.007560 -0.238769 -1.699383 -5.589237   
   bs(pseudotime, df=8)[4]  0.495997  1.066375  6.016636  0.307727 -7.015829   
   bs(pseudotime, df=8)[5]  4.303521 -1.121429  5.843637  3.422356 -5.144053   
   bs(pseudotime, df=8)[6]  5.316972  7.132685  4.903174  4.169022 -6.636533   
   bs(pseudotime, df=8)[7]  5.392619  5.970867  4.848661  4.031403 -5.415489   
   bs(pseudotime, df=8)[8]  4.449085  7.508679  3.878354  3.405123 -5.226408   
   
                                Chga       Cck      Ins1      Nnat      Ins2  \
   Intercept               -4.491948 -1.729480 -0.883534 -1.629171 -2.281118   
   bs(pseudotime

Here is an example where we use a Gaussian on log transformed data.

In [8]:
import numpy as np
from copy import deepcopy

formulas = {
    "mean": [{"variable": "pseudotime", "transform": {"type": "spline", "df": 8}}],
    "sdev": [{"variable": "pseudotime", "transform": {"type": "spline", "df": 3}}],
    "copula": [{"variable": "cell_type"}]
}

example_sce_transform = deepcopy(example_sce)
example_sce_transform.X = np.log1p(example_sce_transform.X)
state = {"template": example_sce_transform, "family": "Gaussian", "formulas": formulas, "n_cells": 2000}
generate_data(state)

Epoch 25/25, Loss: 1.4900, Val Loss: 1.4865


Estimating copula correlation: 100%|██████████| 3/3 [00:00<00:00, 67.17it/s]


({'marginal': {'mean':                               Pyy      Iapp      Chgb      Rbp4      Spp1  \
   Intercept                0.212969  0.210547  0.214014  0.212424  0.218284   
   bs(pseudotime, df=8)[1]  0.183440  0.091343 -0.142998  0.166336  0.229393   
   bs(pseudotime, df=8)[2]  0.165389  0.073122 -0.093624  0.073957  0.224984   
   bs(pseudotime, df=8)[3]  0.164250  0.021182  0.180529  0.072117  0.202226   
   bs(pseudotime, df=8)[4]  0.201119  0.091104  0.215213  0.203539  0.033821   
   bs(pseudotime, df=8)[5]  0.215041  0.206698  0.216888  0.216143  0.002433   
   bs(pseudotime, df=8)[6]  0.216660  0.216910  0.216160  0.216909 -0.015082   
   bs(pseudotime, df=8)[7]  0.217459  0.218656  0.215445  0.217220  0.035787   
   bs(pseudotime, df=8)[8]  0.218359  0.219639  0.216191  0.218320  0.115469   
   
                                Chga       Cck      Ins1      Nnat      Ins2  \
   Intercept                0.213894  0.214254  0.203226  0.206428  0.211410   
   bs(pseudotime